In [1]:
# ==========================================================
# Project: Customer Cohort Retention & CLTV Analysis
# Dataset: Olist Brazilian E-Commerce Dataset
# Author: Yusuf Bahrainwala
# ==========================================================

# Objectives:
# - Understanding the datasets
# - Assess data quality
# - Cleaning and prepareing data
# - Building a master transaction dataset

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [3]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

## Data Loading

In [4]:
customers = pd.read_csv(r"C:\Users\Yusuf\Downloads\Data\Raw\olist_customers_dataset.csv")
orders = pd.read_csv(r"C:\Users\Yusuf\Downloads\Data\Raw\olist_orders_dataset.csv")
order_items = pd.read_csv(r"C:\Users\Yusuf\Downloads\Data\Raw\olist_order_items_dataset.csv")
payments = pd.read_csv(r"C:\Users\Yusuf\Downloads\Data\Raw\olist_order_payments_dataset.csv")
products = pd.read_csv(r"C:\Users\Yusuf\Downloads\Data\Raw\olist_products_dataset.csv")
translation = pd.read_csv(r"C:\Users\Yusuf\Downloads\Data\Raw\product_category_name_translation.csv")


In [5]:
print("customers",customers.shape)
print("orders",orders.shape)
print("order_items",order_items.shape)
print("payments",payments.shape)
print("products",products.shape)
print("translation",translation.shape)

customers (99441, 5)
orders (99441, 8)
order_items (112650, 7)
payments (103886, 5)
products (32951, 9)
translation (71, 2)


## Data Dictionary
| Dataset     | Primary Key              | Foreign Key | Purpose           |
| ----------- | ------------------------ | ----------- | ----------------- |
| Customers   | customer_id              | -           | Customer details  |
| Orders      | order_id                 | customer_id | Order information |
| Order Items | order_id + order_item_id | product_id  | Revenue           |
| Payments    | order_id                 | -           | Payment value     |
| Products    | product_id               | -           | Product details   |
| Translation | product_category_name    | -           | English category  |


## Initial Data Inspection

In [6]:
# customers
print("customers data inspection")
customers.head()
customers.info()
customers.describe(include='all')
customers.columns
customers.sample(5)

customers data inspection
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   customer_id               99441 non-null  object
 1   customer_unique_id        99441 non-null  object
 2   customer_zip_code_prefix  99441 non-null  int64 
 3   customer_city             99441 non-null  object
 4   customer_state            99441 non-null  object
dtypes: int64(1), object(4)
memory usage: 3.8+ MB


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
17872,2c4e6626a5b947f38f954b988fbd7522,e14b02e4cd8aca2d19d7fac116ea5347,81200,curitiba,PR
10578,77b0d15d00d45a25263845e1b5d52833,6ab1d0feda86c0d09edd5dce03db789a,3127,sao paulo,SP
94648,c0357ea4d9212b51cd537ce680f08d55,b9389715e7ac8bf1950d755028d37ed3,85854,foz do iguacu,PR
83070,556228709c357078e1b42f98e97fdd70,e84e318fa505b77fd51fcc2b2a897f84,3642,sao paulo,SP
66568,6096638d1c65231c08ba8d33ff598fda,6c003a48450635a9baa1c84666623117,17210,jau,SP


In [7]:
# orders
print("orders data inspection")
orders.head()
orders.info()
orders.describe(include='all')
orders.columns
orders.sample(5)

orders data inspection
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-null  object
 3   order_purchase_timestamp       99441 non-null  object
 4   order_approved_at              99281 non-null  object
 5   order_delivered_carrier_date   97658 non-null  object
 6   order_delivered_customer_date  96476 non-null  object
 7   order_estimated_delivery_date  99441 non-null  object
dtypes: object(8)
memory usage: 6.1+ MB


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
16297,0ffcdff33eb40b6ce4b6cef424e1c63f,7dacda3efc762f10146d7d824846c015,delivered,2018-06-13 12:37:27,2018-06-13 13:02:17,2018-06-14 07:42:00,2018-06-15 23:50:46,2018-07-18 00:00:00
97829,4294d6c2b09c5f2438c01b11c91ff7a8,fbfb33cc34116ccf95b043f7cd31c692,delivered,2018-06-10 13:42:58,2018-06-10 13:55:13,2018-06-11 13:03:00,2018-07-02 12:41:37,2018-07-16 00:00:00
38583,5f61b71c278e0e65231c70f058f4ab00,6644f96e1da51ce92f80d0c2793199fa,delivered,2018-05-23 20:42:04,2018-05-23 20:57:32,2018-05-28 13:18:00,2018-06-01 15:28:41,2018-06-26 00:00:00
91394,119f1430154997cf2ea4e905667408d1,326bd99b43665ac9e198d45b189740b5,delivered,2018-05-29 02:29:23,2018-05-29 02:51:18,2018-05-29 14:04:00,2018-06-06 20:04:33,2018-07-04 00:00:00
21647,2e346f2848b0263dc1775fa2b5af2719,ce510a4d2db930a570c19134f75a80cf,delivered,2018-06-23 07:13:25,2018-06-23 07:39:27,2018-06-26 15:59:00,2018-07-03 16:44:25,2018-07-19 00:00:00


In [8]:
# order_items
print("order_items data inspection")
order_items.head()
order_items.info()
order_items.describe(include='all')
order_items.columns
order_items.sample(5)

order_items data inspection
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   order_id             112650 non-null  object 
 1   order_item_id        112650 non-null  int64  
 2   product_id           112650 non-null  object 
 3   seller_id            112650 non-null  object 
 4   shipping_limit_date  112650 non-null  object 
 5   price                112650 non-null  float64
 6   freight_value        112650 non-null  float64
dtypes: float64(2), int64(1), object(4)
memory usage: 6.0+ MB


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
78172,b1e54b4e95f856a38246ab5a53b59dbc,1,4f9a85a01a3a49b2760aed83cf3d78e1,1554a68530182680ad5c8b042c3ab563,2017-09-05 22:55:12,179.99,30.30
67575,9a886ac42ca6eb5571fbdd1bdd8be02e,1,89f055104adb9365d7f7b5c475f77742,6b15924333bd1a741595fe981ea04822,2017-04-20 00:34:00,26.90,10.96
24389,378d03578566e65bc195cc5296a31526,1,2604055d4992311bc72b115b6d5adda9,440dd6ab244315c632130ecfb63827b1,2017-09-13 21:30:20,234.00,14.66
80121,b644b094c23f3320d4d8386f40e092b2,1,bb32b03270ad3f5a8aa9474d354119ab,f9ec7093df3a7b346b7bcf7864069ca3,2018-08-22 13:35:11,27.80,15.30
34297,4d9466366a6ea93d8bbd504529c9a1cf,1,72172e982e8b92155069e4201c92c0bb,e9779976487b77c6d4ac45f75ec7afe9,2017-10-01 21:24:45,37.49,9.27


In [9]:
# payments
print("payments data inspection")
payments.head()
payments.info()
payments.describe(include='all')
payments.columns
payments.sample(5)

payments data inspection
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103886 entries, 0 to 103885
Data columns (total 5 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   order_id              103886 non-null  object 
 1   payment_sequential    103886 non-null  int64  
 2   payment_type          103886 non-null  object 
 3   payment_installments  103886 non-null  int64  
 4   payment_value         103886 non-null  float64
dtypes: float64(1), int64(2), object(2)
memory usage: 4.0+ MB


,order_id,payment_sequential,payment_type,payment_installments,payment_value
37646,c0c876d98ac48f4ddb75a5645f6f5016,1,credit_card,1,33.08
35486,445e35406984b81b3f9109de8994e197,1,credit_card,3,102.84
42556,c6c96aae858cf470816e5080de363e47,1,credit_card,1,10.17
44585,c448299f4076c96eb0849f981cf29a2f,1,credit_card,1,43.72
24395,07242eb6740fc0feb5044eda405a4782,1,boleto,1,118.28


In [10]:
# products
print("products data inspection")
products.head()
products.info()
products.describe(include='all')
products.columns
products.sample(5)

products data inspection
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  object 
 1   product_category_name       32341 non-null  object 
 2   product_name_lenght         32341 non-null  float64
 3   product_description_lenght  32341 non-null  float64
 4   product_photos_qty          32341 non-null  float64
 5   product_weight_g            32949 non-null  float64
 6   product_length_cm           32949 non-null  float64
 7   product_height_cm           32949 non-null  float64
 8   product_width_cm            32949 non-null  float64
dtypes: float64(7), object(2)
memory usage: 2.3+ MB


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
6061,2e288007b869cff009a184b071b7c925,brinquedos,38.0,471.0,1.0,600.0,35.0,8.0,20.0
5266,43a24e051b37c8e88accdfdeecddb71b,informatica_acessorios,33.0,253.0,1.0,1000.0,37.0,19.0,19.0
30787,0fc323f82937aa2d350753be764a155f,ferramentas_jardim,43.0,219.0,1.0,1900.0,45.0,30.0,42.0
31523,e8288cf2d07e2b1b0b889132fc8cd8d5,cama_mesa_banho,55.0,677.0,1.0,1300.0,16.0,50.0,40.0
191,224703229a7c966a5b191651e1561be2,esporte_lazer,52.0,234.0,4.0,8150.0,20.0,20.0,20.0


In [11]:
# translation
print("translation data inspection")
translation.head()
translation.info()
translation.describe(include='all')
translation.columns
translation.sample(5)

translation data inspection
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 71 entries, 0 to 70
Data columns (total 2 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   product_category_name          71 non-null     object
 1   product_category_name_english  71 non-null     object
dtypes: object(2)
memory usage: 1.2+ KB


,product_category_name,product_category_name_english
49,construcao_ferramentas_seguranca,construction_tools_safety
23,malas_acessorios,luggage_accessories
27,construcao_ferramentas_jardim,costruction_tools_garden
70,seguros_e_servicos,security_and_services
36,construcao_ferramentas_ferramentas,costruction_tools_tools


In [12]:
orders['order_status'].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

#### "Only orders with order_status = "delivered" will be retained for Cohort Analysis and CLTV calculations because only completed orders represent realized customer purchases and revenue."

In [13]:
orders['order_status'].value_counts(normalize=True) * 100

order_status
delivered      97.020344
shipped         1.113223
canceled        0.628513
unavailable     0.612423
invoiced        0.315765
processing      0.302692
created         0.005028
approved        0.002011
Name: proportion, dtype: float64

In [14]:
# Check duplicates for every dataset
customers.duplicated().sum()

orders.duplicated().sum()

order_items.duplicated().sum()

payments.duplicated().sum()

products.duplicated().sum()

translation.duplicated().sum()

np.int64(0)

In [15]:
customers['customer_id'].duplicated().sum()

np.int64(0)

In [16]:
orders['order_id'].duplicated().sum()

np.int64(0)

In [17]:
products['product_id'].duplicated().sum()

np.int64(0)

In [18]:
order_items[['order_id', 'order_item_id']].duplicated().sum()

np.int64(0)

In [19]:
payments[['order_id', 'payment_sequential']].duplicated().sum()

np.int64(0)

In [20]:
orders_clean = orders[orders['order_status'] == 'delivered'].copy()

print(orders.shape)
print(orders_clean.shape)

(99441, 8)
(96478, 8)


### Date & Time Handling

In [21]:
orders_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 96478 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       96478 non-null  object
 1   customer_id                    96478 non-null  object
 2   order_status                   96478 non-null  object
 3   order_purchase_timestamp       96478 non-null  object
 4   order_approved_at              96464 non-null  object
 5   order_delivered_carrier_date   96476 non-null  object
 6   order_delivered_customer_date  96470 non-null  object
 7   order_estimated_delivery_date  96478 non-null  object
dtypes: object(8)
memory usage: 6.6+ MB


In [22]:
# Convert date columns
date_columns = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_columns:
    orders_clean[col] = pd.to_datetime(orders_clean[col])

orders_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 96478 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       96478 non-null  object        
 1   customer_id                    96478 non-null  object        
 2   order_status                   96478 non-null  object        
 3   order_purchase_timestamp       96478 non-null  datetime64[ns]
 4   order_approved_at              96464 non-null  datetime64[ns]
 5   order_delivered_carrier_date   96476 non-null  datetime64[ns]
 6   order_delivered_customer_date  96470 non-null  datetime64[ns]
 7   order_estimated_delivery_date  96478 non-null  datetime64[ns]
dtypes: datetime64[ns](5), object(3)
memory usage: 6.6+ MB


## Building the Master Transaction Table

In [30]:
fact_sales_df = pd.merge(
    orders_clean,
    customers,
    on="customer_id",
    how="left"
)

In [31]:
fact_sales_df.shape

(96478, 12)

In [32]:
fact_sales_df = pd.merge(
    fact_sales_df,
    order_items,
    on="order_id",
    how="left"
)

In [33]:
print(fact_sales_df.shape)

print("Unique Orders :", fact_sales_df["order_id"].nunique())
print("Missing Products :", fact_sales_df["product_id"].isna().sum())

(110197, 18)
Unique Orders : 96478
Missing Products : 0


In [34]:
fact_sales_df = pd.merge(
    fact_sales_df,
    products,
    on="product_id",
    how="left"
)

In [35]:
print(fact_sales_df.shape)

(110197, 26)


In [36]:
fact_sales_df = pd.merge(
    fact_sales_df,
    translation,
    on="product_category_name",
    how="left"
)

In [37]:
fact_sales_df.shape

(110197, 27)

## Product Name Translation

In [38]:
fact_sales_df['product_category'] = (
    fact_sales_df['product_category_name_english']
    .fillna('Unknown')
    .str.replace('_', ' ', regex=False)
    .str.title()
)

In [41]:
fact_sales_df.shape

(110197, 28)

## Payment Aggregation

In [42]:
payments_clean_df = (
    payments.groupby("order_id", as_index=False)
    .agg(
        payment_value=("payment_value", "sum"),
        payment_installments=("payment_installments", "max"),
        payment_type=("payment_type", lambda x: ", ".join(sorted(x.unique())))
    )
)

In [43]:
payments_clean_df.shape

(99440, 4)

In [44]:
payments_clean_df['order_id'].duplicated().sum()

np.int64(0)

In [47]:
fact_sales_df = pd.merge(
    fact_sales_df,
    payments_clean_df,
    on='order_id',
    how='left'
)

In [48]:
print(fact_sales_df.shape)

(110197, 31)


## Removing unwanted columns

In [50]:
# removing unwanted columns

columns_to_drop = [
    'customer_id',
    'customer_zip_code_prefix',
    'seller_id',
    'shipping_limit_date',
    'product_category_name',
    'product_category_name_english',
    'product_name_lenght',
    'product_description_lenght',
    'product_photos_qty',
    'product_weight_g',
    'product_length_cm',
    'product_height_cm',
    'product_width_cm'
]

fact_sales_df.drop(columns=columns_to_drop, inplace=True)

In [51]:
print(fact_sales_df.shape)
fact_sales_df.info()

(110197, 18)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 110197 entries, 0 to 110196
Data columns (total 18 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   order_id                       110197 non-null  object        
 1   order_status                   110197 non-null  object        
 2   order_purchase_timestamp       110197 non-null  datetime64[ns]
 3   order_approved_at              110182 non-null  datetime64[ns]
 4   order_delivered_carrier_date   110195 non-null  datetime64[ns]
 5   order_delivered_customer_date  110189 non-null  datetime64[ns]
 6   order_estimated_delivery_date  110197 non-null  datetime64[ns]
 7   customer_unique_id             110197 non-null  object        
 8   customer_city                  110197 non-null  object        
 9   customer_state                 110197 non-null  object        
 10  order_item_id                  110197 non-null  int64  

In [52]:
fact_sales_df[fact_sales_df['payment_value'].isna()]

,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_city,customer_state,order_item_id,product_id,price,freight_value,product_category,payment_value,payment_installments,payment_type
34056,bfbd0f9bdef84302105ad712db648a6c,delivered,2016-09-15 12:16:38,2016-09-15 12:16:38,2016-11-07 17:11:53,2016-11-09 07:47:38,2016-10-04,830d5b7aaa3b6f1e9ad63703bec97d23,sao joaquim da barra,SP,1,5a6b04657a4c5ee34285d1e4619a96b4,44.99,2.83,Health Beauty,NaN,NaN,NaN
34057,bfbd0f9bdef84302105ad712db648a6c,delivered,2016-09-15 12:16:38,2016-09-15 12:16:38,2016-11-07 17:11:53,2016-11-09 07:47:38,2016-10-04,830d5b7aaa3b6f1e9ad63703bec97d23,sao joaquim da barra,SP,2,5a6b04657a4c5ee34285d1e4619a96b4,44.99,2.83,Health Beauty,NaN,NaN,NaN
34058,bfbd0f9bdef84302105ad712db648a6c,delivered,2016-09-15 12:16:38,2016-09-15 12:16:38,2016-11-07 17:11:53,2016-11-09 07:47:38,2016-10-04,830d5b7aaa3b6f1e9ad63703bec97d23,sao joaquim da barra,SP,3,5a6b04657a4c5ee34285d1e4619a96b4,44.99,2.83,Health Beauty,NaN,NaN,NaN


In [53]:
payments[payments['order_id'] == 'bfbd0f9bdef84302105ad712db648a6c']

,order_id,payment_sequential,payment_type,payment_installments,payment_value


In [54]:
fact_sales_df = fact_sales_df[
    fact_sales_df["payment_value"].notna()
].copy()

In [55]:
fact_sales_df.shape

(110194, 18)

## Renaming Columns

In [56]:
# renaming columns

fact_sales_df.rename(columns={
    'order_purchase_timestamp': 'purchase_datetime',
    'customer_unique_id': 'customer_key',
    'customer_city': 'city',
    'customer_state': 'state',
    'order_item_id': 'item_id',
    'price': 'item_price',
    'payment_value': 'total_payment'
}, inplace=True)

In [57]:
fact_sales_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 110194 entries, 0 to 110196
Data columns (total 18 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   order_id                       110194 non-null  object        
 1   order_status                   110194 non-null  object        
 2   purchase_datetime              110194 non-null  datetime64[ns]
 3   order_approved_at              110179 non-null  datetime64[ns]
 4   order_delivered_carrier_date   110192 non-null  datetime64[ns]
 5   order_delivered_customer_date  110186 non-null  datetime64[ns]
 6   order_estimated_delivery_date  110194 non-null  datetime64[ns]
 7   customer_key                   110194 non-null  object        
 8   city                           110194 non-null  object        
 9   state                          110194 non-null  object        
 10  item_id                        110194 non-null  int64         
 11  produ

In [58]:
fact_sales_df.reset_index(drop=True, inplace=True)

In [59]:
fact_sales_df.index

RangeIndex(start=0, stop=110194, step=1)

In [61]:
fact_sales_df.to_csv("fact_sales.csv", index=False)

In [62]:
customers.columns

Index(['customer_id', 'customer_unique_id', 'customer_zip_code_prefix',
       'customer_city', 'customer_state'],
      dtype='object')

In [63]:
customers_clean_df = customers.copy()

customers_clean_df = (
    customers_clean_df
    .drop(columns=['customer_id', 'customer_zip_code_prefix'])
    .rename(columns={
        'customer_unique_id': 'customer_key',
        'customer_city': 'city',
        'customer_state': 'state'
    })
)

customers_clean_df.head()

,customer_key,city,state
0,861eff4711a542e4b93843c6dd7febb0,franca,SP
1,290c77bc529b7ac935b93aa66c333dc3,sao bernardo do campo,SP
2,060e732b5b29e8181a18229c7b0b2b5e,sao paulo,SP
3,259dac757896d24d7702b9acbbff3f3c,mogi das cruzes,SP
4,345ecd01c38d18a9036ed96c73b8d066,campinas,SP


In [64]:
customers_clean_df.shape
customers_clean_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   customer_key  99441 non-null  object
 1   city          99441 non-null  object
 2   state         99441 non-null  object
dtypes: object(3)
memory usage: 2.3+ MB


In [65]:
customers_clean_df['customer_key'].duplicated().sum()

np.int64(3345)

In [66]:
customers_clean_df = (
    customers_clean_df
    .drop_duplicates(subset='customer_key')
    .reset_index(drop=True)
)

In [67]:
customers_clean_df['customer_key'].duplicated().sum()

np.int64(0)

In [68]:
customers_clean_df.to_csv(
    "customers_clean.csv",
    index=False
)

In [69]:
orders_clean_df = (
    orders
    .merge(
        customers[['customer_id', 'customer_unique_id']],
        on='customer_id',
        how='left'
    )
)

orders_clean_df = (
    orders_clean_df
    .query("order_status == 'delivered'")
    .drop(columns=['customer_id'])
    .rename(columns={
        'customer_unique_id': 'customer_key',
        'order_purchase_timestamp': 'purchase_datetime'
    })
)

date_columns = [
    'purchase_datetime',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

orders_clean_df[date_columns] = orders_clean_df[date_columns].apply(pd.to_datetime)

In [70]:
orders_clean_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 96478 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       96478 non-null  object        
 1   order_status                   96478 non-null  object        
 2   purchase_datetime              96478 non-null  datetime64[ns]
 3   order_approved_at              96464 non-null  datetime64[ns]
 4   order_delivered_carrier_date   96476 non-null  datetime64[ns]
 5   order_delivered_customer_date  96470 non-null  datetime64[ns]
 6   order_estimated_delivery_date  96478 non-null  datetime64[ns]
 7   customer_key                   96478 non-null  object        
dtypes: datetime64[ns](5), object(3)
memory usage: 6.6+ MB


In [71]:
orders_clean_df.to_csv(
    "orders_clean.csv",
    index=False
)

In [72]:
order_items_clean_df = (
    order_items
    .drop(columns=[
        'seller_id',
        'shipping_limit_date'
    ])
    .rename(columns={
        'order_item_id': 'item_id',
        'price': 'item_price'
    })
)

In [73]:
order_items_clean_df.shape

(112650, 5)

In [74]:
order_items_clean_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 5 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   order_id       112650 non-null  object 
 1   item_id        112650 non-null  int64  
 2   product_id     112650 non-null  object 
 3   item_price     112650 non-null  float64
 4   freight_value  112650 non-null  float64
dtypes: float64(2), int64(1), object(2)
memory usage: 4.3+ MB


In [75]:
order_items_clean_df.duplicated(
    subset=['order_id', 'item_id']
).sum()

np.int64(0)

In [76]:
order_items_clean_df.to_csv(
    "order_items_clean.csv",
    index=False
)

In [78]:
products_clean_df = (
    fact_sales_df[
        ['product_id', 'product_category']
    ]
    .copy()
)

In [79]:
products_clean_df['product_category'] = (
    products_clean_df['product_category']
    .fillna('Unknown')
)

In [80]:
products_clean_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 110194 entries, 0 to 110193
Data columns (total 2 columns):
 #   Column            Non-Null Count   Dtype 
---  ------            --------------   ----- 
 0   product_id        110194 non-null  object
 1   product_category  110194 non-null  object
dtypes: object(2)
memory usage: 1.7+ MB


In [81]:
products_clean_df.shape

(110194, 2)

In [82]:
products_clean_df.to_csv(
    "products_clean.csv",
    index=False
)

In [84]:
payments_clean_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99440 entries, 0 to 99439
Data columns (total 4 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   order_id              99440 non-null  object 
 1   payment_value         99440 non-null  float64
 2   payment_installments  99440 non-null  int64  
 3   payment_type          99440 non-null  object 
dtypes: float64(1), int64(1), object(2)
memory usage: 3.0+ MB


In [85]:
payments_clean_df['order_id'].duplicated().sum()

np.int64(0)

In [86]:
payments_clean = (
    payments_clean_df.rename(columns={
        'payment_value': 'total_payment'
    })
)

In [87]:
payments_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99440 entries, 0 to 99439
Data columns (total 4 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   order_id              99440 non-null  object 
 1   total_payment         99440 non-null  float64
 2   payment_installments  99440 non-null  int64  
 3   payment_type          99440 non-null  object 
dtypes: float64(1), int64(1), object(2)
memory usage: 3.0+ MB


In [88]:
payments_clean.to_csv(
    "payments_clean.csv",
    index=False
)